# Passauer Jahrbücher · Pipeline — Colab Quickstart

Thin notebook that clones the pipeline repo, mounts Drive, edits the volume config, and runs everything.

Tested on **Colab L4 (22.5 GB VRAM)** with the HuggingFace backend. Expect roughly 30-60 s/page; a 350-page volume runs ~3-6 hours end-to-end.

## 1. Clone the repo + install

Replace `Maelkolb` with your GitHub username (or the org/repo where you pushed).

In [ ]:
!git clone https://github.com/Maelkolb/Passauer_Jahrbuecher_Pipeline.git
%cd Passauer_Jahrbuecher_Pipeline

In [ ]:
# [hf] = local Chandra model (the Colab path).
# [vllm] = talk to a remote vLLM server instead (set ocr.vllm_url in the config).
!pip install -q -e ".[hf]"

## 2. Mount Drive (skip if your PDF is already on the Colab filesystem)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 4. Configure + run

Either edit `configs/pjb-048-2006.yaml` directly, or override fields in Python like below.

In [ ]:
from pjb_pipeline.config import VolumeConfig
from pjb_pipeline.pipeline import run

cfg = VolumeConfig.from_yaml('configs/pjb-048-2006.yaml')

# Adjust per-volume:
cfg.pdf_path     = '/content/drive/MyDrive/Passauer_Jahrbuch/Passauer_Jahrbuch_XLVIII_2006_compressed.pdf'
cfg.output_root  = '/content/output'
cfg.page_range   = (3, 20)            # start small while testing; set to None for the whole book

# To use a remote vLLM server instead of local HF:
# cfg.ocr.method = 'vllm'
# cfg.ocr.vllm_url = 'http://your-gpu-box:8000/v1'

run(cfg)

## 5. Preview the HTML edition (optional)

In [ ]:
from IPython.display import HTML, display
from pathlib import Path
import html as _html

html_dir = cfg.html_dir
index_html = (html_dir / 'index.html').read_text(encoding='utf-8')
css_inline = (html_dir / 'assets' / 'edition.css').read_text(encoding='utf-8')
preview = (index_html
    .replace('<link rel="stylesheet" href="assets/edition.css">', f'<style>{css_inline}</style>')
    .replace('<script src="assets/edition.js" defer></script>', ''))
display(HTML(f'<iframe srcdoc="{_html.escape(preview, quote=True)}" '
             f'style="width:100%; height:900px; border:1px solid #ccc; background:#f4ecdb;"></iframe>'))

## 6. Download the bundle

In [ ]:
from google.colab import files
bundle = Path(cfg.output_root) / f'{cfg.slug}.zip'
files.download(str(bundle))

## 7. After processing all volumes — merge graphs

Once you've run the pipeline over every volume, merge the per-volume JSON-LD fragments into a single corpus graph:

```python
!python scripts/merge_graphs.py corpus.jsonld output/*/graph/*.jsonld
```

Same-named authors across volumes collapse to a single `Person` node — that's the free cross-volume linkage.